In [ ]:
# Install required libraries (if not already installed)
!pip install catboost xgboost lightgbm imbalanced-learn keras joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.6 MB/s eta 0:00:00


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
import catboost
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from keras.models import Sequential
from keras.layers import Dense
import joblib


In [ ]:
# Load the dataset from Google Drive
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Path to your dataset in Google Drive
file_path = '/content/drive/MyDrive/Dataset/dataset.csv'

# Load the dataset
df = pd.read_csv(file_path)

# Show the first few rows to inspect the data
df.head()


MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Check data types of each column
df.dtypes

# Check for missing values
df.isnull().sum()


In [ ]:
# Visualize missing values
import seaborn as sns
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()


In [ ]:
# Define numerical columns
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Visualize distributions of numerical features
df[numerical_cols].hist(bins=20, figsize=(12, 10))
plt.tight_layout()
plt.show()


In [ ]:
# Visualize the distribution of categorical features
categorical_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Credit_History', 'Property_Area', 'Loan_Status']

plt.figure(figsize=(12, 6))
for i, col in enumerate(categorical_cols, 1):
    plt.subplot(2, 4, i)
    sns.countplot(data=df, x=col)
    plt.title(f'{col} Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Engineering and Handling Missing Values

# Total Income (ApplicantIncome + CoapplicantIncome)
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']

# Loan-to-Income Ratio (LoanAmount / TotalIncome)
df['LoanToIncomeRatio'] = df['LoanAmount'] / df['TotalIncome']

# Binning the 'LoanAmount' for better interpretation
df['LoanAmountBin'] = pd.cut(df['LoanAmount'], bins=[0, 100, 200, 300, 400, np.inf], labels=['Low', 'Medium', 'High', 'Very High', 'Extreme'])

# Handle missing values
for col in df.columns:
    if df[col].dtype == 'object' or df[col].dtype.name == 'category':
        df[col] = df[col].astype(str)  # Convert categorical columns to string
        df[col] = df[col].fillna(df[col].mode()[0])  # Fill categorical columns with mode
    else:
        df[col] = df[col].fillna(df[col].median())  # Fill numerical columns with median


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical features using Label Encoding
label_encoder = LabelEncoder()

# Apply Label Encoding to categorical columns
cat_features = df.select_dtypes(include='object').columns.tolist()  # List of categorical columns
for col in cat_features:
    df[col] = label_encoder.fit_transform(df[col])  # Transform the column


In [ ]:
# Check for missing values in the target variable (Loan_Status)
print(df['Loan_Status'].isnull().sum())

# If there are missing values, you can either drop rows or fill them
df.dropna(subset=['Loan_Status'], inplace=True)  # Drop rows with missing target values

# After handling missing target values, proceed with train-test split
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Handle class imbalance using SMOTE (Synthetic Minority Over-sampling Technique)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


In [ ]:
# Initialize base models (XGBoost, LightGBM, and CatBoost)

# CatBoost Model
catboost_model = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, cat_features=cat_features, loss_function='Logloss', eval_metric='AUC', early_stopping_rounds=30, random_seed=42, verbose=0)

# XGBoost Model
xgb_model = XGBClassifier(random_state=42, scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]))

# LightGBM Model
lgb_model = lgb.LGBMClassifier(random_state=42, scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]))


In [ ]:
# Hyperparameter Tuning for XGBoost
param_grid_xgb = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'n_estimators': [100, 200, 500]
}

grid_search_xgb = GridSearchCV(XGBClassifier(random_state=42), param_grid_xgb, cv=3, scoring='accuracy', n_jobs=-1)
grid_search_xgb.fit(X_train, y_train)
print("Best Parameters for XGBoost:", grid_search_xgb.best_params_)


In [ ]:
# Step 13: Ensemble Methods (Stacking and Voting)
# Define the base models for stacking
base_learners = [
    ('xgb', xgb_model),
    ('lgbm', lgb_model),
    ('catboost', catboost_model)
]

# Meta-model (Logistic Regression)
meta_model = LogisticRegression()

# Stacking model
stacking_model = StackingClassifier(estimators=base_learners, final_estimator=meta_model)

# Train the stacking model
stacking_model.fit(X_train_smote, y_train_smote)

# Voting Classifier (Soft Voting)
voting_model = VotingClassifier(estimators=[('xgb', xgb_model), ('lgbm', lgb_model), ('catboost', catboost_model)], voting='soft')
voting_model.fit(X_train_smote, y_train_smote)


In [ ]:
# Step 14: Evaluate the Models
models = {'Stacking Model': stacking_model, 'Voting Classifier': voting_model}

for model_name, model in models.items():
    print(f"Evaluating {model_name}...")
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("ROC AUC:", roc_auc_score(y_test, y_proba))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))


In [ ]:
# Step 15: Plot ROC AUC for Stacking Model
from sklearn.metrics import roc_curve

y_pred_prob = stacking_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label="Stacking Model")
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc='lower right')
plt.show()


In [ ]:
# Step 16: Save the Stacking Model
joblib.dump(stacking_model, 'stacking_loan_model.pkl')
print("Model saved as stacking_loan_model.pkl")
